# Step 1: clean and audit January 2024 FHV trips

This notebook preserves raw data and writes every input record to exactly one of
**accepted** or **quarantined**. Demand, simulation, fare-analysis and monetary
eligibility are separate flags: their counts may overlap and must not be added.
Repeated composite keys are flagged, never dropped or called confirmed duplicates.

Only trip cleaning is implemented. Zone-hour/zone-pair aggregation, weather and
holiday integration, forecasts, simulation and profit calculation come later.
The pre-submission report's full-year, exact-duplicate and join claims remain unverified.

Run from the repository root or `notebooks/`, using Python 3.11, Java 17/21 and
`requirements.txt`. Default: validate a bounded sample only. Set
`ADS_RUN_FULL_MONTH=1` in the kernel environment to additionally run January.
Every run uses new output folders; nothing is overwritten.

Last successful full January run: **2026-11-14, 14:25 UTC**, fresh kernel, all cells completed. Runtime versions and audit paths are printed below.
Last successful full Phase 1 execution: 15 November 2026 IST; finalized version `finalized-20261114T190620-6b22f8cb`. See shared handover for verification and reuse.


## 1. Configuration and compatibility

Paths are relative to the repo, not a teammate's home directory. UTC is used only
as a neutral Spark session setting when subtracting the source's timezone-free
timestamps; this step does not convert request times to UTC or local standard time.
Weather-time alignment is a later step. Driver memory and parallelism stay bounded.

In [1]:
# Step 1 of the ETL: configure reproducible trip cleaning without changing raw files.
import csv
import hashlib
import importlib.metadata as metadata
import json
import os
import platform
import re
import subprocess
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "AGENTS.md").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "AGENTS.md").is_file():
    raise RuntimeError("Run the notebook from the repo root or its notebooks folder")
sys.path.insert(0, str(PROJECT_ROOT))

TRIP_PATH = PROJECT_ROOT / "data/raw/trips/fhvhv_tripdata_2024-01.parquet"
LOOKUP_PATH = PROJECT_ROOT / "data/raw/zones/taxi_zone_lookup.csv"
ARCHIVE_PATH = PROJECT_ROOT / "data/raw/zones/taxi_zones.zip"
EXPECTED_MONTH_ROWS = 19_663_930
REAL_SAMPLE_ROWS = 10_000
DRIVER_MEMORY = "3g"
SHUFFLE_PARTITIONS = 48
RUN_FULL_MONTH = os.environ.get("ADS_RUN_FULL_MONTH", "0") == "1"
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ.setdefault("SPARK_LOCAL_IP", "127.0.0.1")

java_result = subprocess.run(["java", "-version"], capture_output=True, text=True, check=True)
java_version = (java_result.stderr or java_result.stdout).splitlines()[0]
java_major = int(re.search(r'version "(\d+)', java_version).group(1))
if sys.version_info < (3, 10) or java_major not in (17, 21):
    raise RuntimeError("This environment needs Python >=3.10 and Java 17 or 21")
VERSIONS = {"python": platform.python_version(), "java": java_version}
for package in ["pyspark", "py4j", "ruff", "pytest", "pyarrow", "nbformat", "nbclient", "ipykernel"]:
    VERSIONS[package] = metadata.version(package)
if VERSIONS["pyspark"] != "4.0.1":
    raise RuntimeError("Use the tested Spark version in requirements.txt")
print(json.dumps(VERSIONS, indent=2))
print("Full-month run enabled:", RUN_FULL_MONTH)

{
  "python": "3.11.9",
  "java": "openjdk version \"21.0.11\" 2026-04-21",
  "pyspark": "4.0.1",
  "py4j": "0.10.9.9",
  "ruff": "0.14.4",
  "pytest": "9.1.1",
  "pyarrow": "24.0.0",
  "nbformat": "5.10.4",
  "nbclient": "0.10.4",
  "ipykernel": "7.2.0"
}
Full-month run enabled: False


## 2. Gate the run on synthetic tests and Ruff

The tests use invented rows, including exact filter boundaries, zero duration,
nulls, non-finite values, repeated keys, and recognised/unrecognised destinations.
Ruff checks Python and notebook code for mistakes such as undefined names. Either
failure stops this notebook before sample or full-month outputs are written.

In [2]:
if os.environ.get("ADS_PREFLIGHT_COMPLETE") == "1":
    print("Tests and Ruff passed in this runner before notebook execution.")
else:
    for arguments in [["-m", "pytest", "tests", "-q"], ["-m", "ruff", "check", "."]]:
        check = subprocess.run(
            [sys.executable, *arguments], cwd=PROJECT_ROOT,
            capture_output=True, text=True,
        )
        print(check.stdout)
        if check.returncode:
            print(check.stderr)
            raise RuntimeError(f"Validation failed: {' '.join(arguments)}")


Tests and Ruff passed in this runner before notebook execution.


## 3. Check inputs and start local Spark

Parquet footer metadata gives the expected January count without collecting the
month. Hashes are checked against our earlier inspection, not a publisher-supplied
checksum. The lookup has unique IDs 1–265; verified boundary coverage is 1–263,
with recognised unmapped IDs 264 and 265. No trip-to-zone join is performed here.

In [3]:
import pyarrow.parquet as parquet
from pyspark.sql import SparkSession



def hash_file(path):
    """Return a file's SHA256 using bounded reads, to verify raw data is unchanged."""
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


INPUT_HASHES = {str(path.relative_to(PROJECT_ROOT)): hash_file(path)
                for path in [TRIP_PATH, LOOKUP_PATH, ARCHIVE_PATH]}
inspection_path = PROJECT_ROOT / "data/inspection/january_2024_initial_inspection.json"
if inspection_path.exists():
    inspection = json.loads(inspection_path.read_text())
    for item in inspection["files"]:
        if INPUT_HASHES[item["path"]] != item["sha256"]:
            raise RuntimeError(f"Raw input changed since inspection: {item['path']}")
else:
    print("No earlier local baseline: current source hashes are recorded, without claiming an earlier audit.")
parquet_rows = parquet.ParquetFile(TRIP_PATH).metadata.num_rows
assert parquet_rows == EXPECTED_MONTH_ROWS
with LOOKUP_PATH.open(newline="") as source:
    known_zone_ids = [int(row["LocationID"]) for row in csv.DictReader(source)]
assert len(known_zone_ids) == len(set(known_zone_ids)) == 265
assert set(known_zone_ids) == set(range(1, 266))

spark = (
    SparkSession.builder.master("local[2]").appName("ADS-January-trip-cleaning")
    .config("spark.driver.memory", DRIVER_MEMORY)
    .config("spark.driver.maxResultSize", "128m")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.ui.enabled", "false")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.shuffle.partitions", str(SHUFFLE_PARTITIONS))
    .config("spark.sql.ansi.enabled", "true")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
raw_trips = spark.read.parquet(str(TRIP_PATH))
raw_trips.printSchema()
print("Input rows from Parquet footer:", parquet_rows)
print("No zone join: categories will use the verified lookup ID set.")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/11/14 21:53:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- originating_base_num: string (nullable = true)
 |-- request_datetime: timestamp_ntz (nullable = true)
 |-- on_scene_datetime: timestamp_ntz (nullable = true)
 |-- pickup_datetime: timestamp_ntz (nullable = true)
 |-- dropoff_datetime: timestamp_ntz (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- trip_miles: double (nullable = true)
 |-- trip_time: long (nullable = true)
 |-- base_passenger_fare: double (nullable = true)
 |-- tolls: double (nullable = true)
 |-- bcf: double (nullable = true)
 |-- sales_tax: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)
 |-- tips: double (nullable = true)
 |-- driver_pay: double (nullable = true)
 |-- shared_request_flag: string (nullable = true)
 |-- shared_match_flag: string (nullable = true)
 |-- access_a_

## 4. Audit definitions

Acceptance depends on present request/pickup times, pickup ID, duration and miles;
finite miles; duration 60–10,800 seconds; speed 1–80 mph; and request before pickup.
Speed is evaluated only for positive `trip_time`. Missing optional data is retained.
Missing destination, drop-off time or service licence does not alone erase otherwise
usable pickup demand. These rows have separate simulation/key-completeness flags.

Both mapped endpoints and usable drop-off timing are needed for simulation.
Monetary eligibility uses base fare and tolls; fare-analysis eligibility uses base
fare alone. Missing/negative/non-finite monetary values remain unchanged. Tips are
never revenue. A mapped pickup here means one of the 263 boundary zones (including
Newark Airport); launch-area selection is not part of cleaning.

The exact collision key is service licence + request/pickup/drop-off timestamps +
pickup/drop-off IDs. Only complete keys are compared, across the whole input scope
before partitioning. Sample collisions describe the sample, not January.

Filter counts form a sequential waterfall. Individual flags overlap; eligibility
counts are independent. Timestamp differences are signed timestamp elapsed seconds
minus `trip_time`, with any nonzero difference flagged but not used as a filter.

In [4]:
print("Audit definitions are shared in src/trip_audit.py; the month runner below uses them.")

Audit definitions are shared in src/trip_audit.py; the month runner below uses them.


## 5. Run one scope, reconcile and save without overwriting

Spark stores annotated trips on disk during audits instead of collecting the month
in Python RAM. The single output dataset is partitioned by `cleaning_status`:
accepted and quarantined are disjoint and exhaustive. Every repeated-key row is
written too. Read-back counts must match input counts and independent eligibility
counts. Audit JSON records versions, scope and hashes. Failed runs keep their output
folders for investigation; they are not marked complete.

In [5]:
from src.trip_audit import run_cleaning_scope as run_month_cleaning


def run_cleaning_scope(raw_scope, expected_rows, scope_name):
    """Run the shared audit definitions with this notebook configuration."""
    return run_month_cleaning(
        raw_scope, expected_rows, scope_name, project_root=PROJECT_ROOT,
        known_zone_ids=known_zone_ids, versions=VERSIONS, input_hashes=INPUT_HASHES,
    )

## 6. Bounded real-data validation

Only 10,000 raw rows are passed to sample cleaning. All sample reconciliation
assertions and saved-output checks must pass before the full-month cell can run.
This is a bounded check of real input, not a representative statistical sample.
Synthetic tests cover edge cases that this slice might not contain.

In [6]:
sample_audit = run_cleaning_scope(raw_trips.limit(REAL_SAMPLE_ROWS), REAL_SAMPLE_ROWS, "sample")
assert sample_audit["status"] == "complete"
print("Sample validation passed. No full-month result is inferred from sample rates.")

26/11/14 21:53:32 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Saved: data/processed/trip_cleaning/sample-20261114T162330-aebd5658/trips
Audit: data/audits/trip_cleaning/sample-20261114T162330-aebd5658.json
Partition: {'accepted': 9921, 'quarantined': 79}
Independent eligibility: {'is_demand_eligible': 9921, 'is_simulation_eligible': 9509, 'is_fare_analysis_eligible': 9921, 'is_monetary_eligible': 9921}
Repeated-key records: 0
Filter waterfall: {'has_missing_core_field': 10000, 'has_nonfinite_trip_miles': 10000, 'has_nonpositive_trip_seconds': 10000, 'has_duration_outside_limits': 9998, 'has_speed_outside_limits': 9992, 'has_invalid_request_order': 9921}
Timestamp differences: {'min_signed_seconds': -1.0, 'max_signed_seconds': 0.0, 'mean_signed_seconds': -0.0004, 'signed_p50_p90_p99': [0.0, 0.0, 0.0], 'absolute_p50_p90_p99': [0.0, 0.0, 0.0], 'negative_differences': 4, 'positive_differences': 0, 'unavailable': 0, 'exact_match': 9996, 'above_0_through_1_second': 4, 'above_1_through_60_seconds': 0, 'above_60_through_300_seconds': 0, 'above_300_second

## 7. January run (only after tests, Ruff and sample checks pass)

The full run must reconcile all 19,663,930 input records. Its audit reproduces only
this month's results under the explicitly agreed key and cleaning rules. It does
not retroactively verify the pre-submission report's different, undocumented audit.

In [7]:
month_audit = None
if RUN_FULL_MONTH:
    assert sample_audit["status"] == "complete"
    month_audit = run_cleaning_scope(raw_trips, EXPECTED_MONTH_ROWS, "january-2024")
    assert sum(month_audit["partition_counts"].values()) == EXPECTED_MONTH_ROWS
else:
    print("Full month skipped. Set ADS_RUN_FULL_MONTH=1 to enable the approved January run.")

Full month skipped. Set ADS_RUN_FULL_MONTH=1 to enable the approved January run.


## 8. Verify raw preservation and finish

The hashes must still match. The final message is printed only after the run
reaches this cell; the audit and notebook outputs show which scope actually ran.
Do not commit any files in `data/`. This notebook stops after cleaning.

In [8]:
for path in [TRIP_PATH, LOOKUP_PATH, ARCHIVE_PATH]:
    assert hash_file(path) == INPUT_HASHES[str(path.relative_to(PROJECT_ROOT))]
spark.stop()
print("Raw hashes unchanged. Notebook reached the end without errors.")
print("Completed scopes:", ["sample", "january-2024"] if month_audit else ["sample"])

Raw hashes unchanged. Notebook reached the end without errors.
Completed scopes: ['sample']
